# Per-season re-prep: filtered + resized + prompt-changed, with verification log

Downloads seasons **one at a time** from `SharpaIT/Robotic_Origami_Challenge`,
applies the same filtering/resizing `trex_origami.prepare` already does, with a
changed task prompt, and writes each season to its own directory named
`<season>_<transform tag>`. Stream-and-delete: the raw season is dropped after
conversion, so peak disk stays at `output-so-far + one season`, matching
`trex_origami/fetch.py`'s own design.

**Reused directly, not reimplemented** (see each cell for exactly which
function): `trex_origami.fetch.season_allow_patterns/have_season/local_season_dir/drop_season`,
`trex_origami.prepare.PrepConfig/prepare_season/write_dataset_meta`,
`trex_origami.accel.install`, `trex_origami.stats.compute_and_write`,
`trex_origami.verify.verify`.

Transforms applied, and why each one:
1. **No `head_right`, no `tactile_raw`** — not a new filter, this is what
   `VIDEO_KEYS` (`trex_origami/seasons.py:32`) already excludes at download
   time (`season_allow_patterns`), reused as-is. Nothing to add here.
2. **Resize to 224x224** — `PrepConfig.image_size`, already the default,
   passed through explicitly for clarity.
3. **Prompt changed** — `PrepConfig.instruction`, replacing the current
   `INSTRUCTION = "north ces task"` (`trex_origami/seasons.py:24`, not real
   natural language) with a descriptive one. Stage-aware prompting
   (`phase_mode="progress"`) is deliberately **not** enabled here — `phase`
   is pure linear time-position (`prepare.py:352`), not a real fold-boundary
   label, so it would inject a confidently-wrong stage claim (see
   `checklist.md` item 3). Left at `phase_mode="none"`.
4. **Optional, off by default: head crop** (§ near the bottom) — the one
   "what else" candidate from `recipe_alignment.md` item A2. Reference T-Rex
   crops `(0,300,140,540)` on a 640x360 frame before resizing; origami's
   camera is 480x480, so that box does **not** transfer — needs fresh
   coordinates from a real frame, not a guess. Off by default for that reason.

**One thing this surfaced that's worth fixing separately**: `fetch.py`'s own
`download_season()` never pins a `revision`, but the working example this
notebook is based on used `revision="competition-paper-set"` explicitly. If
the hub's default branch has since moved, `trex_origami`'s existing pipeline
may not be training on the pinned competition snapshot. This notebook adds
the missing pin locally (next cell) rather than editing `fetch.py`, since
that's existing pipeline code this task didn't ask to change.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv


## 1. Setup


In [ ]:
import os

PROJECT_ROOT = "/content/origami_trex"

if not os.path.isdir(PROJECT_ROOT):
    token = ""
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_PAT") or ""
    except Exception:
        pass
    token = token or os.environ.get("GITHUB_PAT", "")
    auth = f"{token}@" if token else ""
    !git clone https://{auth}github.com/KshitijBhat/origami_trex.git {PROJECT_ROOT}
else:
    print(f"already present: {PROJECT_ROOT}")


In [ ]:
%%bash
apt-get -qq install -y ffmpeg > /dev/null
command -v uv >/dev/null 2>&1 || pip install -q uv
UV_PY="$(command -v python)"
uv pip install --python "${UV_PY}" -q -e /content/origami_trex/T-Rex


## 2. Hugging Face token

Read from a Colab secret, not typed into a cell — same reasoning as the
`GITHUB_PAT` handling in the other notebooks in this project: avoids putting a
live credential into a file that might get saved or shared.

Add it once: left sidebar -> key icon -> secrets -> new secret named
`HF_TOKEN`, value = your token. Needs read access to
`SharpaIT/Robotic_Origami_Challenge`.


In [ ]:
from google.colab import userdata

HF_TOKEN = None
try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    pass
if not HF_TOKEN:
    print("No HF_TOKEN secret found -- add one (see markdown above) if the "
          "dataset repo needs auth, or leave as None if it's public for you.")


## 3. Parameters

`SEASONS_IN_REVISION` is discovered dynamically from the Hub at `REVISION`
(next cell) via a shallow `list_repo_tree` call, so a pinned revision's
*actual* season set is what gets used instead of blindly trusting
`seasons.py`'s recorded 101/25 split. `SEASONS` defaults to a **small
2-season test slice** of that discovered list — expand to
`SEASONS_IN_REVISION` (full sweep) only after a test run's verification log
looks right; that's a multi-hour, large-output run and not something to
default to on first execution.

The original hardcoded `TRAIN_SEASONS`/`VAL_SEASONS` import is still there
(commented alternative in the next cell), kept as a fallback rather than
removed. Splitting the *discovered* list into train/val is a follow-up step,
not done here yet — the next cell just reports how many discovered seasons
land in each recorded bucket, since a pinned revision can legitimately hold a
different set of seasons than whatever `seasons.py` was written against.


In [ ]:
import sys
sys.path.insert(0, "/content/origami_trex/T-Rex")
from trex_origami.seasons import TRAIN_SEASONS, VAL_SEASONS, HF_REPO_ID

REVISION = os.environ.get("HF_REVISION", "competition-paper-set")
# override without editing this cell: run `%env HF_REVISION=<other-tag>` in a
# cell above this one, then re-run this cell -- same env-var-override
# convention every .sh script in this project uses (TRAIN_BSZ, ACTION_HORIZON, ...).
REPO_ID = HF_REPO_ID

def list_seasons_in_revision(repo_id, revision, token=None):
    """`season_*` top-level folders actually present at `revision`.

    One shallow `list_repo_tree` call (non-recursive) instead of
    `list_repo_files`, which would walk every video/parquet file in the whole
    repo just to read directory names.
    """
    from huggingface_hub import HfApi
    from huggingface_hub.hf_api import RepoFolder
    api = HfApi(token=token)
    entries = api.list_repo_tree(repo_id, repo_type="dataset", revision=revision,
                                 recursive=False)
    return sorted(e.path for e in entries
                  if isinstance(e, RepoFolder) and e.path.startswith("season_"))

SEASONS_IN_REVISION = list_seasons_in_revision(REPO_ID, REVISION, token=HF_TOKEN)

# How the discovered list lines up with the recorded split (`seasons.py`,
# sourced from `dataset.md`) -- printed rather than assumed, since a pinned
# revision can legitimately hold a different season set than whatever ref
# `seasons.py` was written against.
_train_set, _val_set = set(TRAIN_SEASONS), set(VAL_SEASONS)
_in_train = [s for s in SEASONS_IN_REVISION if s in _train_set]
_in_val = [s for s in SEASONS_IN_REVISION if s in _val_set]
_unsplit = [s for s in SEASONS_IN_REVISION if s not in _train_set and s not in _val_set]
print(f"{len(SEASONS_IN_REVISION)} season(s) found in revision '{REVISION}'")
print(f"  {len(_in_train)} match TRAIN_SEASONS, {len(_in_val)} match VAL_SEASONS, "
      f"{len(_unsplit)} not in either recorded split")

# Small test slice by default. Swap in SEASONS_IN_REVISION for the full sweep
# over everything this revision actually has.
SEASONS = SEASONS_IN_REVISION[:2]
# SEASONS = SEASONS_IN_REVISION   # <-- full sweep, uncomment when ready

# Original hardcoded split (kept, not removed) -- use this instead if you want
# the recorded 101/25 split regardless of what this revision holds (will fail
# per-season on any entry not actually present at REVISION, see counts above).
# SEASONS = TRAIN_SEASONS[:2]
# SEASONS = TRAIN_SEASONS + VAL_SEASONS   # <-- full sweep, uncomment when ready

# Train/val split of SEASONS (whatever subset is actually queued this run,
# not necessarily all of SEASONS_IN_REVISION) -- N_VAL directly controls how
# many of the queued seasons go to val; the rest go to train. Deterministic
# (sorted then seeded-shuffled), so reruns with the same SEASONS + SPLIT_SEED
# reproduce the same split. Set N_VAL explicitly for exact control, e.g.
# N_VAL = 5, rather than relying on the fraction default below.
import random
N_VAL = max(1, round(0.2 * len(SEASONS))) if len(SEASONS) > 1 else 0
SPLIT_SEED = 0
_shuffled = sorted(SEASONS)
random.Random(SPLIT_SEED).shuffle(_shuffled)
VAL_SPLIT, TRAIN_SPLIT = _shuffled[:N_VAL], _shuffled[N_VAL:]
print(f"split: {len(TRAIN_SPLIT)} train season(s), {len(VAL_SPLIT)} val season(s) (N_VAL={N_VAL})")

NEW_INSTRUCTION = "fold the paper into an origami airplane"   # replaces "north ces task"
IMAGE_SIZE = 224
SAMPLE_STRIDE = 5
ACTION_CHUNK = 25
VQVAE_WINDOW = 16

# Optional, unvalidated -- see §5 near the bottom before turning this on.
APPLY_HEAD_CROP = False
HEAD_CROP_BOX = None   # (top, bottom, left, right) in the RAW frame's own pixels -- None until measured

TRANSFORM_TAG = f"_noHR_noTR_{IMAGE_SIZE}"
if APPLY_HEAD_CROP:
    TRANSFORM_TAG += "_cropped"

CACHE_ROOT = "/content/_season_cache"      # raw download scratch, dropped per season
OUT_ROOT = "/content/drive/MyDrive/iros2026/origami_reprep"
SPLIT_ROOT = os.path.join(OUT_ROOT, f"reprep{TRANSFORM_TAG}")
TRAIN_ROOT = os.path.join(SPLIT_ROOT, "train")
VAL_ROOT = os.path.join(SPLIT_ROOT, "val")
LOG_PATH = os.path.join(SPLIT_ROOT, "reprep_log.json")
os.makedirs(TRAIN_ROOT, exist_ok=True)
os.makedirs(VAL_ROOT, exist_ok=True)

print(f"{len(SEASONS)} season(s) queued, tag='{TRANSFORM_TAG}'")
print(f"output -> {TRAIN_ROOT}  /  {VAL_ROOT}")


### 3a. How many episodes is that?

Season count isn't episode count -- seasons vary in length. Reuses
`trex_origami.prepare.read_episode_specs` (the exact function `prepare_season`
itself calls) against a **meta-only** download: `meta/episodes/**/*.parquet`
per season is tens/hundreds of KB, versus ~1GB/season for the full
video+data tree, so this is cheap to run over the whole discovered list, not
just the queued slice.


In [ ]:
from trex_origami.prepare import read_episode_specs, _season_root

def count_episodes(seasons, repo_id, revision, cache_root, token=None):
    """Episode count per season, from `meta/episodes/**` only (no video/data)."""
    from huggingface_hub import snapshot_download
    counts = {}
    for season in seasons:
        snapshot_download(
            repo_id=repo_id, repo_type="dataset", revision=revision,
            allow_patterns=[f"{season}/lerobot3.0/meta/episodes/**"],
            local_dir=cache_root, token=token,
        )
        counts[season] = len(read_episode_specs(_season_root(cache_root, season), season))
    return counts

EPISODE_COUNT_CACHE = os.path.join(CACHE_ROOT, "_episode_count_scratch")
episode_counts = count_episodes(SEASONS_IN_REVISION, REPO_ID, REVISION,
                                EPISODE_COUNT_CACHE, token=HF_TOKEN)

total_all = sum(episode_counts.values())
total_queued = sum(episode_counts[s] for s in SEASONS)
print(f"{total_all} episode(s) total across {len(SEASONS_IN_REVISION)} season(s) "
      f"in revision '{REVISION}'")
print(f"{total_queued} episode(s) in the {len(SEASONS)} season(s) currently queued (SEASONS)")


## 4. Per-season download with the missing revision pin

Mirrors `trex_origami.fetch.download_season` exactly (same `local_dir`
convention, same `season_allow_patterns` filter, same post-download
completeness check via `have_season`) -- the only difference is threading
`revision` through to `snapshot_download`, which the original doesn't accept
at all. Everything else here is imported, not copied.


In [ ]:
from trex_origami.fetch import (season_allow_patterns, local_season_dir,
                                have_season, drop_season)

def download_season_pinned(season, cache_root, repo_id=REPO_ID, revision=None,
                           token=None, max_workers=8):
    from huggingface_hub import snapshot_download
    target = local_season_dir(cache_root, season)
    os.makedirs(cache_root, exist_ok=True)
    snapshot_download(
        repo_id=repo_id, repo_type="dataset", revision=revision,
        allow_patterns=season_allow_patterns(season),
        local_dir=cache_root, token=token, max_workers=max_workers,
    )
    if not have_season(cache_root, season):
        raise RuntimeError(f"{season}: download finished but the expected "
                          f"lerobot3.0 tree is missing under {target}")
    return target


## 5. Optional: head-view workspace crop (off by default)

The one extra transform worth considering, from `recipe_alignment.md` item A2:
reference T-Rex crops the head view to the workspace *before* resizing
(`(0,300,140,540)` on their 640x360 camera); origami currently resizes the
full frame. With only 64 LLM tokens per view at 224x224, a crop would spend
that fixed budget on the fold region instead of background -- but the
reference numbers don't transfer (different camera, different resolution).

**Do not enable this with `HEAD_CROP_BOX = None`** -- it's a no-op guard, not
a default. Before turning it on: decode one raw `head_left` frame from a
downloaded season (`ffprobe`/`cv2.imread` on a single frame) and read off
real pixel bounds for *this* camera's workspace region, then set
`HEAD_CROP_BOX` in §3 accordingly.

Implemented as a local monkeypatch of `prepare.decode_frames`, not an edit to
`prepare.py` itself -- this stays entirely inside this notebook. It only
intercepts the `head_left` stream (matched by path, since `decode_frames`
doesn't otherwise know which camera it's decoding); wrist/deform go through
the unpatched original unchanged. Reuses `prepare._select_expr` /
`prepare._split_jpegs` directly rather than re-deriving the ffmpeg command.


In [ ]:
import subprocess
import trex_origami.prepare as _prep_mod

if APPLY_HEAD_CROP:
    if HEAD_CROP_BOX is None:
        raise ValueError("APPLY_HEAD_CROP=True but HEAD_CROP_BOX is None -- "
                         "measure real crop bounds first, see markdown above.")
    _orig_decode_frames = _prep_mod.decode_frames

    def _decode_frames_maybe_cropped(video_path, ranges, expected, *, scale, quality):
        if "head_left" not in video_path:
            return _orig_decode_frames(video_path, ranges, expected, scale=scale, quality=quality)
        top, bottom, left, right = HEAD_CROP_BOX
        w, h = right - left, bottom - top
        vf = f"select='{_prep_mod._select_expr(ranges)}',crop={w}:{h}:{left}:{top}"
        if scale:
            vf += f",scale={scale}:{scale}:flags=lanczos"
        cmd = ["ffmpeg", "-nostdin", "-v", "error", "-i", video_path, "-vf", vf,
              "-vsync", "0", "-f", "image2pipe", "-c:v", "mjpeg", "-q:v", str(quality), "pipe:1"]
        proc = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        if proc.returncode != 0:
            raise RuntimeError(f"ffmpeg (cropped) failed on {video_path}: "
                              f"{proc.stderr.decode('utf-8', 'replace')[:2000]}")
        return _prep_mod._split_jpegs(proc.stdout, expected, os.path.basename(video_path))

    _prep_mod.decode_frames = _decode_frames_maybe_cropped
    print(f"HEAD CROP enabled: box={HEAD_CROP_BOX} (top,bottom,left,right) -- UNVALIDATED, "
          f"inspect a few output frames before trusting this")
else:
    print("HEAD CROP disabled (default)")


## 6. Main loop -- one season at a time, stream-and-delete

Per season: download (pinned revision) -> convert (`prepare_season`, GPU
decode via `accel.install()`) -> merge this season's episodes into its split's
shared `TRAIN_ROOT`/`VAL_ROOT` (`meta/dataset.json`, unioned via
`_merge_entries` -- same helper `prepare_fast.py`'s own multi-season loop
uses, so many seasons land in one root instead of each getting its own) ->
log -> drop the raw season regardless of outcome. Once every season in
`SEASONS` is done, norm stats are fit **once**, on the accumulated `TRAIN_ROOT`
only, and copied to `VAL_ROOT` -- not per-season, since a per-season stats fit
would only see a fraction of the real train set. `verify()` then runs once per
split root.


In [ ]:
import time
from trex_origami import accel
from trex_origami.prepare import PrepConfig, prepare_season, write_dataset_meta, _merge_entries
from trex_origami.stats import compute_stats, write_stats
from trex_origami.verify import verify

accel.install()

log_entries = []
for season in SEASONS:
    t0 = time.time()
    split = "val" if season in VAL_SPLIT else "train"
    split_root = VAL_ROOT if split == "val" else TRAIN_ROOT
    entry = {"season": season, "split": split, "out_dir": split_root}
    try:
        download_season_pinned(season, CACHE_ROOT, revision=REVISION, token=HF_TOKEN)

        cfg = PrepConfig(sample_stride=SAMPLE_STRIDE, action_chunk=ACTION_CHUNK,
                         image_size=IMAGE_SIZE, vqvae_window=VQVAE_WINDOW,
                         instruction=NEW_INSTRUCTION, phase_mode="none")
        entries = prepare_season(season, CACHE_ROOT, split_root, cfg, overwrite=True)
        # Merge into the split root's shared meta/dataset.json (union by "file",
        # same helper prepare_fast.py's own multi-season loop uses) rather than
        # overwriting -- each season here adds to the same train/val root, not
        # its own directory.
        write_dataset_meta(split_root, _merge_entries(split_root, entries), cfg)

        n_samples = sum(e["n_samples"] for e in entries)
        entry.update(status="ok", n_episodes=len(entries), n_samples=n_samples,
                     elapsed_s=round(time.time() - t0, 1))
        print(f"[{season}] -> {split}: {len(entries)} episodes, {n_samples} samples "
              f"({entry['elapsed_s']:.0f}s)")
    except Exception as e:
        entry.update(status="error", error=f"{type(e).__name__}: {e}",
                     elapsed_s=round(time.time() - t0, 1))
        print(f"[{season}] FAILED: {entry['error']}")
    finally:
        drop_season(CACHE_ROOT, season)
    log_entries.append(entry)

with open(LOG_PATH, "w") as f:
    json.dump(log_entries, f, indent=2)
n_ok = sum(1 for e in log_entries if e["status"] == "ok")
print(f"\n{n_ok}/{len(log_entries)} season(s) OK -- log: {LOG_PATH}")

# Norm stats fit on train only, copied to val -- same convention as
# run_prepare_fast.sh ("val must normalise with the *train* statistics, or its
# loss is not comparable to the training loss"). Run once here, after every
# season, not per-season -- per-season stats would each be fit on a fraction
# of the real train set.
if any(e["status"] == "ok" and e["split"] == "train" for e in log_entries):
    stats = compute_stats(TRAIN_ROOT, subsample=4)
    write_stats(TRAIN_ROOT, stats)
    write_stats(VAL_ROOT, stats)
    print("norm_stats.json fit on train, copied to val")
else:
    print("no train-split season succeeded -- norm_stats.json not written")

for name, root in (("train", TRAIN_ROOT), ("val", VAL_ROOT)):
    if not os.path.exists(os.path.join(root, "meta", "dataset.json")):
        print(f"[verify {name}] skipped -- root is empty")
        continue
    problems = verify(root, n_samples=8)
    print(f"[verify {name}] {'OK' if not problems else f'{len(problems)} PROBLEM(S)'}")
    for p in problems:
        print(f"    - {p}")


## 7. Verification summary


In [ ]:
import json as _json

with open(LOG_PATH) as f:
    _log = _json.load(f)

print(f"{'season':<45} {'split':<6} {'status':<14} {'episodes':>9} {'samples':>9} {'s':>6}")
print("-" * 97)
for e in _log:
    print(f"{e['season']:<45} {e.get('split', '-'):<6} {e['status']:<14} "
          f"{e.get('n_episodes', '-'):>9} {e.get('n_samples', '-'):>9} "
          f"{e.get('elapsed_s', '-'):>6}")

failed = [e for e in _log if e["status"] != "ok"]
if failed:
    print(f"\n{len(failed)} season(s) need attention -- see {LOG_PATH} for details.")


## 8. Inspect what the model actually sees -- Z random samples

Loads the real `OrigamiDataset` against one of the season directories just
written, picks `Z` random rows, and reports exactly what a training batch
looks like: shapes, ranges, episode time, and normalization before/after.

**Reused, not reimplemented**: `qwen_vla.origami_dataset.OrigamiDataset` for
loading (`__getitem__` for the raw per-row fields -- real PIL decode, real
tactile/deform arrays) and its module-level `_normalize` (the exact function
`collate_fn` itself calls) for the post-normalization numbers. `processor` is
passed as `None` -- checked it's only ever touched inside `collate_fn`
(building tokenized text/vision inputs), never in `__init__`/`_build_index`/
`__getitem__`, so this works without downloading a Qwen3-VL processor for a
purely numeric/visual inspection that doesn't need one.

`deform` has no q01/q99 normalization path in this dataset class at all --
its numbers below are just the raw `_deform()` output (`/255.0` grayscale,
already what `DeformEncoder` consumes directly), not a bug in the "post"
column being the same as "pre" for that field.


In [ ]:
import random

Z = 5
INSPECT_SEED = None   # set an int for reproducible sampling; None = different each run

# TRAIN_ROOT/VAL_ROOT are now shared across all seasons in that split (§6), not
# one directory per season -- default to train (what the model actually
# trains on); point INSPECT_ROOT at VAL_ROOT instead to inspect val.
_ok = [e for e in log_entries if e["status"] == "ok"]
assert _ok, "no successfully-processed season to inspect -- run §6 first, or set INSPECT_ROOT manually"
INSPECT_ROOT = TRAIN_ROOT if any(e["split"] == "train" for e in _ok) else VAL_ROOT
print("inspecting:", INSPECT_ROOT)


In [ ]:
from qwen_vla.origami_dataset import OrigamiDataset, _normalize
from trex_origami.prepare import PrepConfig as _PC

class _Printer:
    # OrigamiDataset's 3rd arg only needs .print() -- stands in for the
    # accelerate Accelerator it otherwise expects, same helper eval_origami.py
    # defines for the same reason.
    @staticmethod
    def print(*args, **kwargs):
        print(*args, **kwargs)

_inspect_cfg = _PC(sample_stride=SAMPLE_STRIDE, action_chunk=ACTION_CHUNK,
                   image_size=IMAGE_SIZE, vqvae_window=VQVAE_WINDOW)
inspect_ds = OrigamiDataset(vars(_inspect_cfg), None, _Printer(), root=INSPECT_ROOT, _quiet=True)

rng = random.Random(INSPECT_SEED)
sample_indices = rng.sample(range(len(inspect_ds)), min(Z, len(inspect_ds)))

samples = []
for idx in sample_indices:
    ei, r = inspect_ds.index[idx]
    ep = inspect_ds.episodes[ei]
    episode_time_s = r * inspect_ds.sample_stride / inspect_ds.src_fps
    row = inspect_ds[idx]
    samples.append({"idx": idx, "episode_index": ei, "row_in_episode": r,
                    "episode_file": ep["file"], "season": ep.get("season", "?"),
                    "episode_time_s": round(episode_time_s, 2), "row": row})
    print(f"idx={idx:>6}  ep={ep['file']}  row_in_ep={r:>4}  t={episode_time_s:6.2f}s  "
          f"task='{row.get('task', '')}'")

In [ ]:
import numpy as np

def _stat(x):
    x = np.asarray(x, dtype=np.float64)
    return {"shape": list(x.shape), "min": float(x.min()), "max": float(x.max()),
           "mean": float(x.mean())}

report = []
for s in samples:
    row = s["row"]
    state_pre = row["state"]
    action_pre = row["action"]
    tacf6_pre = row.get("tacf6_hist")
    deform = row.get("deform")

    state_post = _normalize(state_pre, inspect_ds.state_mask, inspect_ds.state_min, inspect_ds.state_max)
    action_post = _normalize(action_pre, inspect_ds.action_mask, inspect_ds.action_min, inspect_ds.action_max)
    entry = {
        "idx": s["idx"], "episode_file": s["episode_file"],
        "episode_time_s": s["episode_time_s"], "task": row.get("task", ""),
        "state": {"pre": _stat(state_pre), "post": _stat(state_post)},
        "action_chunk": {"pre": _stat(action_pre), "post": _stat(action_post)},
    }
    if tacf6_pre is not None:
        tacf6_post = _normalize(tacf6_pre.reshape(-1, 6), inspect_ds.tacf6_mask,
                                inspect_ds.tacf6_min, inspect_ds.tacf6_max)
        entry["tacf6_hist"] = {"pre": _stat(tacf6_pre), "post": _stat(tacf6_post)}
    if deform is not None:
        entry["deform"] = {"pre": _stat(deform),
                           "post": "n/a -- DeformEncoder consumes this directly, no q01/q99 path"}
    report.append(entry)
    print(f"\\n[{s['idx']}] {s['episode_file']} @ {s['episode_time_s']}s")
    for field in ("state", "action_chunk", "tacf6_hist", "deform"):
        if field in entry:
            pre, post = entry[field]["pre"], entry[field]["post"]
            if isinstance(post, str):
                post_str = post
            else:
                post_str = f"[min={post['min']:.4f} max={post['max']:.4f} mean={post['mean']:.4f}]"
            print(f"  {field:<12} shape={pre['shape']}  "
                  f"pre[min={pre['min']:.4f} max={pre['max']:.4f} mean={pre['mean']:.4f}]  "
                  f"post={post_str}")

### Visualize the images


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(len(samples), 4, figsize=(16, 4 * len(samples)))
if len(samples) == 1:
    axes = axes[None, :]
for i, s in enumerate(samples):
    row = s["row"]
    for j, cam in enumerate(("head", "wrist_left", "wrist_right")):
        axes[i, j].imshow(row[cam])
        axes[i, j].set_title(f"{cam}  [{s['idx']}] t={s['episode_time_s']}s", fontsize=8)
        axes[i, j].axis("off")
    deform = row.get("deform")
    if deform is not None:
        axes[i, 3].imshow(deform.reshape(2, 5, 240, 240).transpose(0, 2, 1, 3).reshape(480, 1200),
                          cmap="gray", vmin=0, vmax=1)
        axes[i, 3].set_title("deform (reassembled 2x5 grid)", fontsize=8)
    axes[i, 3].axis("off")
fig.tight_layout()
plt.show()

In [ ]:
DATASET_LOG_PATH = os.path.join(OUT_ROOT, "dataset_extracted.log")
with open(DATASET_LOG_PATH, "w") as f:
    f.write(f"root: {INSPECT_ROOT}\\nZ: {Z}  seed: {INSPECT_SEED}\\n\\n")
    for entry in report:
        f.write(f"[{entry['idx']}] {entry['episode_file']} @ {entry['episode_time_s']}s  "
               f"task='{entry['task']}'\\n")
        for field in ("state", "action_chunk", "tacf6_hist", "deform"):
            if field not in entry:
                continue
            pre, post = entry[field]["pre"], entry[field]["post"]
            f.write(f"  {field}: shape={pre['shape']}\\n")
            f.write(f"    pre-norm : min={pre['min']:.5f} max={pre['max']:.5f} mean={pre['mean']:.5f}\\n")
            if isinstance(post, str):
                f.write(f"    post-norm: {post}\\n")
            else:
                f.write(f"    post-norm: min={post['min']:.5f} max={post['max']:.5f} mean={post['mean']:.5f}\\n")
        f.write("\\n")
print(f"wrote {DATASET_LOG_PATH}")

## Is FLARE actually being used? (recap, not new code)

No -- confirmed earlier this session, restated here since it's directly
relevant to what this notebook writes into each season's `meta/dataset.json`:
`train_origami.sh` sets `flare_loss_weight=0.0`, and
`OrigamiDataset.bake_flare = use_flare and flare_weight > 0 and n_flare_steps > 0`
(`qwen_vla/origami_dataset.py:189`) evaluates to `False` as a result. That
means future frames are never even read from the parquet for FLARE, no
`flare_pixel_values` are built, and `train.py`'s own `train_flare` gate
(`train_flare = bool(use_flare and args.flare_loss_weight > 0)`, line 1107)
skips both the extra hidden-state capture and the loss computation entirely.
The 32 flare query tokens still exist in the sequence and get gradient
*indirectly* through joint attention on the real action loss, but nothing
about their own stated objective is ever trained. See `checklist.md` item 2.
This notebook doesn't change that (`phase_mode`/`use_flare` aren't touched
here) -- flagging it because it's easy to assume "flare is on" from
`use_flare=1` alone without checking the weight.
